In [2]:
from math import ceil

def calculate_vllm_vram(model, sequence_length, batch_size, precision_bytes):
    # 1. Base Model Memory
    base_memory = model.parameters * precision_bytes * model.overhead_factor

    # 2. KV-Cache with vLLM optimizations
    # Block allocation
    block_size = model.vllm_optimizations.block_size  # typically 16
    blocks_needed = ceil(sequence_length / block_size)
    effective_sequence_length = blocks_needed * block_size

    # GQA support
    kv_heads = model.architecture.kv_heads or model.architecture.attention_heads
    head_dim = model.architecture.head_dim or (model.architecture.hidden_size / model.architecture.attention_heads)

    # Core KV calculation
    kv_cache_base = (2 *                           # keys + values
                     model.architecture.layers *   # transformer layers
                     kv_heads *                     # KV heads (not attention heads for GQA!)
                     head_dim *                     # dimension per head
                     effective_sequence_length *   # tokens (block-aligned)
                     batch_size *                   # concurrent requests
                     precision_bytes)               # bytes per parameter

    # Memory pool overhead
    memory_pool_overhead = kv_cache_base * model.vllm_optimizations.memory_pool_overhead
    kv_cache_total = kv_cache_base + memory_pool_overhead

    # 3. Activation Memory
    activation_multiplier = model.vram_requirements.activation_multiplier  # typically 1.5
    activation_memory = (model.architecture.hidden_size *
                        sequence_length *
                        batch_size *
                        precision_bytes *
                        activation_multiplier)

    # 4. System Overhead
    system_overhead = (base_memory + kv_cache_total) * 0.1

    # Total VRAM
    total_vram = base_memory + kv_cache_total + activation_memory + system_overhead

    return total_vram

# Example usage with stub model object
class ModelStub:
    def __init__(self):
        self.parameters = 7_240_000_000  # 7.24B parameters (Mistral 7B)
        self.overhead_factor = 1.12      # Model loading overhead

        # Architecture configuration
        self.architecture = type('Architecture', (), {
            'layers': 32,                # Number of transformer layers
            'hidden_size': 4096,         # Hidden dimension size
            'attention_heads': 32,       # Number of attention heads
            'kv_heads': 8,              # Number of KV heads (GQA: fewer than attention heads)
            'head_dim': 128             # Dimension per attention head
        })()

        # vLLM optimization settings
        self.vllm_optimizations = type('VLLMOptimizations', (), {
            'block_size': 16,                    # Tokens per memory block
            'memory_pool_overhead': 0.15         # Pre-allocation overhead (15%)
        })()

        # VRAM requirement settings
        self.vram_requirements = type('VRAMRequirements', (), {
            'activation_multiplier': 1.4         # Inference activation scaling
        })()

In [3]:
# Example calculation
model = ModelStub()
sequence_length = 500        # Total tokens (input + output)
batch_size = 1              # Number of concurrent requests
precision_bytes = 2         # fp16 precision (2 bytes per parameter)

total_vram_bytes = calculate_vllm_vram(model, sequence_length, batch_size, precision_bytes)
total_vram_gb = total_vram_bytes / (1024**3)

print(f"Total VRAM required: {total_vram_gb:.2f} GB")

Total VRAM required: 16.70 GB
